In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/datasets', exist_ok=True)

Mounted at /content/drive


In [3]:
import pandas as pd
import glob
import numpy as np
import ast
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report
from sklearn.utils import resample
import tensorflow as tf
import csv
from tensorflow.keras.layers import Input, Embedding, GlobalAveragePooling1D, Dense, Dropout, TextVectorization, GlobalMaxPooling1D, Conv1D, SpatialDropout1D, Concatenate
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.regularizers import l2
import nltk
from nltk.corpus import stopwords
import re

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [4]:
all_files = glob.glob("/content/drive/MyDrive/datasets/*.csv")

df = pd.concat([
    pd.read_csv(f, engine='python', quoting=csv.QUOTE_MINIMAL, on_bad_lines='skip')
    for f in all_files
], ignore_index=True)

print(f"Total rows: {len(df)}")
print(f"Dari {len(all_files)} file: {[os.path.basename(f) for f in all_files]}")
df.info()

Total rows: 18372
Dari 3 file: ['Linkedin_finalfix_1105.csv', 'Jobstreet Cleaned.csv', 'Data_Merged.csv']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18372 entries, 0 to 18371
Data columns (total 19 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   id                18169 non-null  float64
 1   title             18169 non-null  object 
 2   search_role       18372 non-null  object 
 3   job_description   18337 non-null  object 
 4   extracted_skills  18372 non-null  object 
 5   skills_count      18372 non-null  int64  
 6   search_role_raw   5902 non-null   object 
 7   job_level         5902 non-null   object 
 8   company           15834 non-null  object 
 9   location          15835 non-null  object 
 10  location_raw      5902 non-null   object 
 11  salary_display    5902 non-null   object 
 12  salary_min        1667 non-null   float64
 13  salary_max        1667 non-null   float64
 14  salary_avg        1667 non-n

In [5]:
df = df.drop_duplicates()
df = df.drop(columns=['salary', 'search_role_raw', 'job_level', 'location_raw',
                       'salary_display', 'salary_min', 'salary_max', 'salary_avg', 'company', 'location', 'job_url', 'search_location', 'scraped_at'],
             errors='ignore')
print(f"Total duplicated rows: {df.duplicated().sum()}")
df.dropna(inplace=True)
print(df.isnull().sum())

Total duplicated rows: 388
id                  0
title               0
search_role         0
job_description     0
extracted_skills    0
skills_count        0
dtype: int64


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 17998 entries, 0 to 18371
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   id                17998 non-null  float64
 1   title             17998 non-null  object 
 2   search_role       17998 non-null  object 
 3   job_description   17998 non-null  object 
 4   extracted_skills  17998 non-null  object 
 5   skills_count      17998 non-null  int64  
dtypes: float64(1), int64(1), object(4)
memory usage: 984.3+ KB


In [7]:
def parse_skills(val):
    try:
        result = ast.literal_eval(val)
        return ' '.join(result) if isinstance(result, list) and len(result) > 0 else ''
    except:
        return ''

df['extracted_skills_clean'] = df['extracted_skills'].apply(parse_skills)

# FIX: pakai skills aja, jangan gabung title + job_description
df['text_input'] = df['extracted_skills_clean']
df = df[df['text_input'].str.strip() != ''].reset_index(drop=True)

print(f"Total rows: {len(df)}")
print(f"Skills kosong: {(df['text_input'] == '').sum()} rows")
print(df['search_role'].value_counts())

Total rows: 17551
Skills kosong: 0 rows
search_role
Backend Developer       2334
Data Engineer           1698
Software Engineer       1405
Data Scientist          1299
Fullstack Developer     1263
DevOps Engineer         1240
Frontend Developer      1097
Developer               1000
Software                1000
Full stack Developer     998
Software Developer       997
IT                       970
Data Analyst             887
Web Developer            831
ML Engineer              309
Progammer                223
Name: count, dtype: int64


In [8]:
min_samples = 900
role_counts = df['search_role'].value_counts()
valid_roles = role_counts[role_counts >= min_samples].index
df = df[df['search_role'].isin(valid_roles)].reset_index(drop=True)
print(df['search_role'].value_counts())

search_role
Backend Developer       2334
Data Engineer           1698
Software Engineer       1405
Data Scientist          1299
Fullstack Developer     1263
DevOps Engineer         1240
Frontend Developer      1097
Software                1000
Developer               1000
Full stack Developer     998
Software Developer       997
IT                       970
Name: count, dtype: int64


In [9]:
df['search_role'] = df['search_role'].replace({'Full stack Developer': 'Fullstack Developer'})

roles_to_drop = ['Software Engineer', 'Web Developer', 'IT Support', 'IT',
                 'Developer', 'Software', 'System Analyst', 'Progammer', 'Software Developer']
df = df[~df['search_role'].isin(roles_to_drop)]

min_samples = 200
role_counts = df['search_role'].value_counts()
valid_roles = role_counts[role_counts >= min_samples].index
df = df[df['search_role'].isin(valid_roles)].reset_index(drop=True)

print(df['search_role'].value_counts())

search_role
Backend Developer      2334
Fullstack Developer    2261
Data Engineer          1698
Data Scientist         1299
DevOps Engineer        1240
Frontend Developer     1097
Name: count, dtype: int64


In [10]:
encoder = LabelEncoder()
df['label'] = encoder.fit_transform(df['search_role'])
num_classes = len(encoder.classes_)

# text_input sudah di-set di cell sebelumnya, tidak perlu di-set ulang
df['text_input'] = df['text_input'].fillna('').astype(str)

print(f"Classes: {encoder.classes_}")
print(f"Num classes: {num_classes}")

Classes: ['Backend Developer' 'Data Engineer' 'Data Scientist' 'DevOps Engineer'
 'Frontend Developer' 'Fullstack Developer']
Num classes: 6


In [11]:
stop_words_en = set(stopwords.words('english'))
stop_words_id = set(stopwords.words('indonesian'))

all_stop_words = stop_words_en.union(stop_words_id)

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', ' ', text)

    words = text.split()
    clean_words = [w for w in words if w not in all_stop_words]

    return ' '.join(clean_words)

df['text_input'] = df['text_input'].apply(clean_text)

In [12]:
print("=== Label alignment check ===")
for i in range(5):
    print(f"Role: {df['search_role'].iloc[i]}")
    print(f"Label: {df['label'].iloc[i]}")
    print(f"Title: {df['title'].iloc[i]}")
    print()

=== Label alignment check ===
Role: Data Scientist
Label: 2
Title: Data Scientist

Role: Data Scientist
Label: 2
Title: Data Scientist (Remote)

Role: Data Scientist
Label: 2
Title: Data Scientist - AI/ML | Remote

Role: Data Scientist
Label: 2
Title: Data Scientist

Role: Data Scientist
Label: 2
Title: Data Science Analyst (Indonesia)



In [13]:
# 1. Split data asli dulu
X_train, X_val, y_train, y_val = train_test_split(
    df['text_input'], df['label'],
    test_size=0.2, random_state=42, stratify=df['label']
)

# 2. Gabungin sementara buat resample
train_data = pd.DataFrame({'text': X_train, 'label': y_train})

target_samples = 1500
dfs_resampled = []

# 3. FIX: oversampling HANYA kelas minoritas, mayoritas dibiarkan
for role_label in train_data['label'].unique():
    df_role = train_data[train_data['label'] == role_label]
    if len(df_role) < target_samples:
        df_role = resample(df_role, replace=True, n_samples=target_samples, random_state=42)
    dfs_resampled.append(df_role)

# 4. Gabungin dan shuffle
train_data_balanced = pd.concat(dfs_resampled).sample(frac=1, random_state=42).reset_index(drop=True)

X_train_bal = train_data_balanced['text']
y_train_bal = train_data_balanced['label']

print("Distribusi setelah resampling:")
print(pd.Series(y_train_bal).value_counts().sort_index())
print(f"\nTotal training samples: {len(X_train_bal)}")
print(f"Total val samples: {len(X_val)}")

# 5. Vectorizer adapt dari training data
vectorizer = TextVectorization(
    max_tokens=8000,
    output_mode='int',
    output_sequence_length=150,  # diturunin karena input cuma skills, bukan full text
    ngrams=2
)
vectorizer.adapt(X_train_bal.to_numpy())
print(f"Vocab size: {len(vectorizer.get_vocabulary())}")

Distribusi setelah resampling:
label
0    1867
1    1500
2    1500
3    1500
4    1500
5    1809
Name: count, dtype: int64

Total training samples: 9676
Total val samples: 1986
Vocab size: 1985


In [14]:
print("=== Sample text inputs ===")
for role in df['search_role'].unique():
    sample = df[df['search_role'] == role]['text_input'].iloc[0]
    print(f"\n[{role}]")
    print(sample[:200])

print("\n=== Vectorizer vocab sample ===")
print(vectorizer.get_vocabulary()[:50])
print(f"\nVocab size: {len(vectorizer.get_vocabulary())}")

=== Sample text inputs ===

[Data Scientist]
numpy pandas python r scikit learn sql

[Data Engineer]
express power bi python sql tableau

[Backend Developer]
git python solid

[Frontend Developer]
r

[Fullstack Developer]
go python

[DevOps Engineer]
aws azure bash ci cd gcp linux python

=== Vectorizer vocab sample ===
['', '[UNK]', np.str_('r'), np.str_('sql'), np.str_('git'), np.str_('java'), np.str_('scala'), np.str_('python'), np.str_('javascript'), np.str_('aws'), np.str_('react'), np.str_('agile'), np.str_('postgresql'), np.str_('mysql'), np.str_('scala r'), np.str_('docker'), np.str_('ios'), np.str_('azure'), np.str_('java r'), np.str_('gcp'), np.str_('kubernetes'), np.str_('linux'), np.str_('php'), np.str_('c'), np.str_('rust'), np.str_('typescript'), np.str_('r sql'), np.str_('r javascript'), np.str_('golang'), np.str_('java scala'), np.str_('vue'), np.str_('kafka'), np.str_('scrum'), np.str_('javascript r'), np.str_('sql postgresql'), np.str_('r git'), np.str_('android'), np

In [15]:
class CustomEarlyStopping(tf.keras.callbacks.Callback):
    def __init__(self, patience=5):
        super().__init__()
        self.patience = patience
        self.best_weights = None
        self.best_loss = np.inf
        self.wait = 0

    def on_epoch_end(self, epoch, logs=None):
        current_loss = logs.get('val_loss')
        if current_loss < self.best_loss:
            self.best_loss = current_loss
            self.wait = 0
            self.best_weights = self.model.get_weights()
        else:
            self.wait += 1
            if self.wait >= self.patience:
                print(f"\n[INFO] val_loss tidak membaik selama {self.patience} epoch. Stop training!")
                self.model.stop_training = True
                print("[INFO] Mengembalikan bobot model ke epoch terbaik.")
                self.model.set_weights(self.best_weights)

callbacks = [
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1),
    CustomEarlyStopping(patience=5)
]

In [16]:
vocab_size = len(vectorizer.get_vocabulary())

inputs = Input(shape=(1,), dtype=tf.string, name='text_input')
x = vectorizer(inputs)
x = Embedding(input_dim=vocab_size, output_dim=64)(x)
x = SpatialDropout1D(0.3)(x)           # naik dari 0.1
x = Conv1D(128, 5, activation='relu')(x)
x = GlobalMaxPooling1D()(x)
x = Dense(64, activation='relu', kernel_regularizer=l2(0.001))(x)  # tambah L2
x = Dropout(0.5)(x)                    # naik dari 0.3

outputs = Dense(num_classes, activation='softmax', name='role_output')(x)

model = Model(inputs=inputs, outputs=outputs, name="Job_Role_Classifier")
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

print("Mulai proses training...")
history = model.fit(
    X_train_bal.to_numpy(), y_train_bal,
    validation_data=(X_val.to_numpy(), y_val),
    epochs=40,
    batch_size=64,
    callbacks=callbacks
)

Model: "Job_Role_Classifier"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_input (InputLayer)         │ (None, 1)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ text_vectorization              │ (None, 150)            │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, 150, 64)        │       127,040 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d               │ (None, 150, 64)        │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 146, 128)       │        41,088 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d            │ (None, 128)            │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ role_output (Dense)             │ (None, 6)              │           390 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 176,774 (690.52 KB)

 Trainable params: 176,774 (690.52 KB)

 Non-trainable params: 0 (0.00 B)

Mulai proses training...
Epoch 1/40
152/152 ━━━━━━━━━━━━━━━━━━━━ 10s 14ms/step - accuracy: 0.2743 - loss: 1.7010 - val_accuracy: 0.3943 - val_loss: 1.4255 - learning_rate: 0.0010
Epoch 2/40
152/152 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.4588 - loss: 1.3116 - val_accuracy: 0.4502 - val_loss: 1.2193 - learning_rate: 0.0010
Epoch 3/40
152/152 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.5098 - loss: 1.1570 - val_accuracy: 0.4532 - val_loss: 1.1898 - learning_rate: 0.0010
Epoch 4/40
152/152 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - accuracy: 0.5326 - loss: 1.0947 - val_accuracy: 0.4859 - val_loss: 1.1710 - learning_rate: 0.0010
Epoch 5/40
152/152 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.5457 - loss: 1.0509 - val_accuracy: 0.4527 - val_loss: 1.1786 - learning_rate: 0.0010
Epoch 6/40
152/152 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - accuracy: 0.5556 - loss: 1.0300 - val_accuracy: 0.4471 - val_loss: 1.1872 - learning_rate: 0.0010
Epoch 7/40
150/152 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accu

In [17]:
y_pred = np.argmax(model.predict(X_val.to_numpy()), axis=1)
print(classification_report(y_val, y_pred, target_names=encoder.classes_))

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
                     precision    recall  f1-score   support

  Backend Developer       0.40      0.11      0.17       467
      Data Engineer       0.42      0.84      0.56       340
     Data Scientist       0.68      0.48      0.56       260
    DevOps Engineer       0.62      0.54      0.58       248
 Frontend Developer       0.46      0.71      0.56       219
Fullstack Developer       0.49      0.48      0.48       452

           accuracy                           0.49      1986
          macro avg       0.51      0.53      0.48      1986
       weighted avg       0.50      0.49      0.45      1986



In [18]:
import joblib
from google.colab import files

model.save('job_role_model.keras')
print("[INFO] Model berhasil disimpan")

joblib.dump(encoder, 'label_encoder.pkl')
print("[INFO] Label encoder berhasil disimpan")

# Test inference dengan input skill aja (simulasi kondisi production)
loaded_model = tf.keras.models.load_model('job_role_model.keras')
loaded_encoder = joblib.load('label_encoder.pkl')

def rekomendasi_role(skills_text):
    input_tensor = tf.constant([skills_text], dtype=tf.string)
    pred_probs = loaded_model.predict(input_tensor, verbose=0)
    pred_class_idx = np.argmax(pred_probs)
    predicted_role = loaded_encoder.inverse_transform([pred_class_idx])[0]
    confidence = float(np.max(pred_probs))
    return predicted_role, confidence

# Test dengan skill aja, TANPA title/job desc
skill_input = "experience building REST APIs Laravel FastAPI managing databases PostgreSQL"
role, conf = rekomendasi_role(skill_input)
print(f"\n--- Hasil Inference ---")
print(f"Skill Input : {skill_input}")
print(f"Cocok untuk : {role} ({conf:.1%})")

[INFO] Model berhasil disimpan
[INFO] Label encoder berhasil disimpan

--- Hasil Inference ---
Skill Input : experience building REST APIs Laravel FastAPI managing databases PostgreSQL
Cocok untuk : Fullstack Developer (59.5%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>